# Avances III y IV · Detección de fraude con tarjeta de crédito
**Redes neuronales tabulares frente a Gradient Boosting**

Especialización en Inteligencia Artificial, Universidad de Cundinamarca.

Este notebook implementa y prueba los modelos diseñados en el Avance I, usando el mismo pipeline de datos del Avance II. Está dividido en dos partes:

- **Avance III, implementación y pruebas iniciales.** Se entrenan las versiones v1 de cada modelo tal como se diseñaron, se miden y se analizan los errores.
- **Avance IV, rediseño y resultados.** Se aplican las mejoras (ajuste de hiperparámetros, embeddings numéricos para la red, varias semillas y ensamble) y se hace la comparación final con pruebas estadísticas.

**Cómo ejecutarlo**

1. Menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*.
2. Menú *Entorno de ejecución → Ejecutar todas*.
3. Si el notebook no encuentra los datos, pedirá subir `creditcard.csv` o el `.zip` que lo contiene. También se puede arrastrar el archivo al panel de archivos antes de ejecutar.
4. Al terminar se descarga solo `resultados_avances_3_4.zip`, que contiene métricas, predicciones, historiales de entrenamiento y modelos.

Con GPU la ejecución completa toma alrededor de una hora. Para una prueba corta se puede poner `MODO_RAPIDO = True` en la celda de configuración.

## 1. Instalación de librerías
XGBoost, LightGBM y PyTorch ya vienen en Colab, solo falta Optuna para el ajuste de hiperparámetros y pytorch-tabnet para TabNet.

In [ ]:
!pip -q install optuna pytorch-tabnet

## 2. Configuración general
Todos los parámetros del experimento están aquí, igual que en `config/config.yaml` del repositorio.

In [ ]:
import os, io, json, time, copy, math, random, zipfile, platform, warnings, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                             confusion_matrix, precision_score, recall_score, f1_score)
warnings.filterwarnings("ignore")

MODO_RAPIDO = False   # True para una prueba corta de unos 10 minutos

CONFIG = {
    "semilla": 42,
    "proporcion_entrenamiento": 0.70,
    "proporcion_validacion": 0.15,
    "costo_fn": 1.0,          # costo de dejar pasar un fraude
    "costo_fp": 0.02,         # costo de revisar una transaccion legitima (1 fraude = 50 alertas)
    "bootstrap": 300 if MODO_RAPIDO else 1000,
    "trials_xgb": 6 if MODO_RAPIDO else 30,
    "trials_red": 5 if MODO_RAPIDO else 20,
    "minutos_max_optuna_red": 8 if MODO_RAPIDO else 25,
    "epocas_max": 12 if MODO_RAPIDO else 40,
    "epocas_max_optuna": 10 if MODO_RAPIDO else 30,
    "sigmas_embedding": [0.5] if MODO_RAPIDO else [0.1, 0.5, 1.0],
    "semillas_finales": [42, 7] if MODO_RAPIDO else [42, 7, 2024],
}

SALIDA = "/content/resultados" if os.path.isdir("/content") else "resultados"
os.makedirs(SALIDA, exist_ok=True)
INICIO = time.time()

# Paleta de las graficas del notebook
COLORES = {"logreg": "#64748B", "xgb": "#0F766E", "lgb": "#14B8A6", "mlp": "#D97706",
           "tabnet": "#B45309", "emb": "#9333EA", "ens": "#BE123C"}

def fijar_semilla(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError:
        pass

def _json(o):
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return str(o)

RESULTADOS = {"config": CONFIG, "modelos": {}, "historias": {}, "extra": {}, "errores": {}}
PREDICCIONES = {}

def guardar():
    # Se guarda despues de cada modelo, asi un fallo no borra lo anterior.
    with open(f"{SALIDA}/resultados.json", "w", encoding="utf-8") as f:
        json.dump(RESULTADOS, f, indent=1, ensure_ascii=False, default=_json)
    datos = {f"{k}__{p}": v[p] for k, v in PREDICCIONES.items() for p in v}
    np.savez_compressed(f"{SALIDA}/predicciones.npz", **datos)

def registrar_error(etapa, e):
    print(f"ERROR en {etapa}: {e}")
    RESULTADOS["errores"][etapa] = repr(e)
    guardar()

try:
    import torch
    DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
    GPU = torch.cuda.get_device_name(0) if DISPOSITIVO == "cuda" else "sin GPU"
except ImportError:
    torch, DISPOSITIVO, GPU = None, "cpu", "sin GPU"
print("Dispositivo:", DISPOSITIVO, "|", GPU, "| modo rapido:", MODO_RAPIDO)

## 3. Carga de datos
Se busca primero el archivo en la carpeta de Colab, después se intenta descargar desde Kaggle y, si nada de eso funciona, se pide subirlo.

In [ ]:
def leer(ruta):
    if ruta.endswith(".zip"):
        with zipfile.ZipFile(ruta) as z:
            nombre = next(n for n in z.namelist() if n.endswith(".csv") and "__MACOSX" not in n)
            with z.open(nombre) as f:
                return pd.read_csv(f)
    return pd.read_csv(ruta)

def cargar_datos():
    for ruta in ["/content/creditcard.csv", "creditcard.csv", "/content/creditcard_csv.zip",
                 "creditcard_csv.zip", "/content/creditcardfraud.zip", "data/raw/creditcard.csv"]:
        if os.path.exists(ruta):
            print("Leyendo", ruta)
            return leer(ruta)
    try:
        import kagglehub
        carpeta = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
        print("Descargado desde Kaggle en", carpeta)
        return leer(os.path.join(carpeta, "creditcard.csv"))
    except Exception as e:
        print("No se pudo descargar desde Kaggle:", e)
    from google.colab import files
    print("Sube creditcard.csv o el .zip que lo contiene")
    subido = files.upload()
    return leer(next(iter(subido)))

df = cargar_datos()
print(df.shape)

## 4. Pipeline de datos (el mismo del Avance II)
Validación, eliminación de duplicados, partición temporal 70 / 15 / 15, variables derivadas y escalado robusto ajustado solo con entrenamiento.

In [ ]:
COLS_V = [f"V{i}" for i in range(1, 29)]

def validar(df):
    esperadas = ["Time"] + COLS_V + ["Amount", "Class"]
    faltan = [c for c in esperadas if c not in df.columns]
    assert not faltan, f"Faltan columnas: {faltan}"
    assert int(df.isnull().sum().sum()) == 0, "Hay valores nulos"
    assert set(df["Class"].unique()) == {0, 1}, "La clase no es binaria"
    return {"filas": len(df), "duplicados": int(df.duplicated().sum()), "fraudes": int(df["Class"].sum())}

def particion_temporal(df):
    df = df.drop_duplicates().sort_values("Time", kind="mergesort").reset_index(drop=True)
    n = len(df)
    c1 = int(n * CONFIG["proporcion_entrenamiento"])
    c2 = int(n * (CONFIG["proporcion_entrenamiento"] + CONFIG["proporcion_validacion"]))
    return df.iloc[:c1], df.iloc[c1:c2], df.iloc[c2:]

def derivar(parte):
    X = parte[COLS_V].copy()
    hora = (parte["Time"] // 3600) % 24
    X["hora_sin"] = np.sin(2 * np.pi * hora / 24)    # la hora es circular: 23 y 0 quedan cerca
    X["hora_cos"] = np.cos(2 * np.pi * hora / 24)
    X["monto_log"] = np.log1p(parte["Amount"])
    return X

RESULTADOS["extra"]["validacion"] = validar(df)
entrena, valida, prueba = particion_temporal(df)
COLUMNAS = list(derivar(entrena).columns)
N_VARIABLES = len(COLUMNAS)

escalador = RobustScaler().fit(derivar(entrena))          # solo con entrenamiento
X_tr = escalador.transform(derivar(entrena)).astype(np.float32)
X_va = escalador.transform(derivar(valida)).astype(np.float32)
X_te = escalador.transform(derivar(prueba)).astype(np.float32)
y_tr = entrena["Class"].to_numpy().astype(np.int64)
y_va = valida["Class"].to_numpy().astype(np.int64)
y_te = prueba["Class"].to_numpy().astype(np.int64)
montos_te = prueba["Amount"].to_numpy()
hora_te = ((prueba["Time"] // 3600) % 24).to_numpy()
RAZON = float((y_tr == 0).sum() / (y_tr == 1).sum())       # legitimas por cada fraude en entrenamiento

particion = {n: {"filas": int(len(y)), "fraudes": int(y.sum()), "tasa_pct": round(100 * y.mean(), 4)}
             for n, y in [("entrenamiento", y_tr), ("validacion", y_va), ("prueba", y_te)]}
RESULTADOS["extra"]["particion"] = particion
RESULTADOS["extra"]["razon_desbalance"] = RAZON
print(pd.DataFrame(particion).T)
esperado = {"entrenamiento": (198608, 366), "validacion": (42559, 55), "prueba": (42559, 52)}
for n, (f, fr) in esperado.items():
    if (particion[n]["filas"], particion[n]["fraudes"]) != (f, fr):
        print(f"AVISO: {n} no coincide con el Avance II, se esperaban {f} filas y {fr} fraudes")
print("Variables de entrada:", N_VARIABLES, "| razon de desbalance:", round(RAZON, 1))

## 5. Funciones de evaluación
Todas las métricas se calculan igual para todos los modelos. El umbral se escoge en validación minimizando el costo de los errores y se aplica sin cambios en prueba. Los intervalos de confianza salen de un bootstrap estratificado, que remuestrea los fraudes y las legítimas por separado.

In [ ]:
CFN, CFP = CONFIG["costo_fn"], CONFIG["costo_fp"]

def umbral_por_costo(y, p):
    # Recorre todos los umbrales posibles de una vez, ordenando por probabilidad.
    orden = np.argsort(-p, kind="mergesort")
    ps, ys = p[orden], y[orden]
    vp = np.cumsum(ys)
    fp = np.cumsum(1 - ys)
    costo = (ys.sum() - vp) * CFN + fp * CFP
    validos = np.where(np.r_[ps[1:] != ps[:-1], True])[0]   # ultimo indice de cada grupo de empates
    k = validos[np.argmin(costo[validos])]
    return float(ps[k])

def metricas(y, p, umbral, montos=None):
    pred = (p >= umbral).astype(int)
    vn, fp, fn, vp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    orden = np.argsort(-p)
    r = {"auprc": average_precision_score(y, p), "roc_auc": roc_auc_score(y, p),
         "precision": precision_score(y, pred, zero_division=0), "recall": recall_score(y, pred, zero_division=0),
         "f1": f1_score(y, pred, zero_division=0), "umbral": umbral,
         "vp": vp, "fp": fp, "fn": fn, "vn": vn, "alertas": vp + fp, "costo": fn * CFN + fp * CFP,
         "precision_top50": y[orden[:50]].mean(), "precision_top100": y[orden[:100]].mean()}
    if montos is not None:
        r["monto_detectado_pct"] = 100 * montos[(y == 1) & (pred == 1)].sum() / montos[y == 1].sum()
    return {k: float(v) for k, v in r.items()}

def _indices_bootstrap(y, n, semilla):
    rng = np.random.default_rng(semilla)
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    for _ in range(n):
        yield np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])

def bootstrap(y, p, umbral, n=CONFIG["bootstrap"], semilla=42):
    auprc, rec, prec = [], [], []
    for i in _indices_bootstrap(y, n, semilla):
        yy, pp = y[i], p[i]
        pred = pp >= umbral
        auprc.append(average_precision_score(yy, pp))
        rec.append(pred[yy == 1].mean())
        prec.append(yy[pred].mean() if pred.any() else 0.0)
    ic = lambda a: [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]
    return {"auprc_ic95": ic(auprc), "recall_ic95": ic(rec), "precision_ic95": ic(prec)}

def bootstrap_pareado(y, pa, pb, n=CONFIG["bootstrap"], semilla=42):
    # Mismas muestras para los dos modelos: mide si la diferencia de AUPRC es real o es ruido.
    d = np.array([average_precision_score(y[i], pa[i]) - average_precision_score(y[i], pb[i])
                  for i in _indices_bootstrap(y, n, semilla)])
    return {"diferencia_media": float(d.mean()), "ic95": [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))],
            "prob_a_mejor": float((d > 0).mean())}

NOMBRES = {}
def registrar(clave, nombre, p_va, p_te, **extra):
    u = umbral_por_costo(y_va, p_va)
    RESULTADOS["modelos"][clave] = {"nombre": nombre, "val": metricas(y_va, p_va, u),
                                    "test": metricas(y_te, p_te, u, montos_te),
                                    "ic_test": bootstrap(y_te, p_te, u), **extra}
    PREDICCIONES[clave] = {"val": np.asarray(p_va, dtype=np.float32), "test": np.asarray(p_te, dtype=np.float32)}
    NOMBRES[clave] = nombre
    guardar()
    m = RESULTADOS["modelos"][clave]
    print(f"{nombre:<28} AUPRC val {m['val']['auprc']:.4f} | prueba {m['test']['auprc']:.4f} "
          f"| recall {m['test']['recall']:.3f} | precision {m['test']['precision']:.3f}")

def latencia_ms(predictor, X, repeticiones=7):
    # Milisegundos para puntuar 1.000 transacciones en CPU (mediana de varias corridas).
    lote = X[:1000]
    predictor(lote)
    tiempos = []
    for _ in range(repeticiones):
        t0 = time.perf_counter(); predictor(lote); tiempos.append(time.perf_counter() - t0)
    return float(np.median(tiempos) * 1000)

def tabla(fase=None):
    filas = []
    for k, m in RESULTADOS["modelos"].items():
        if fase and m.get("fase") != fase: continue
        t = m["test"]
        filas.append({"modelo": m["nombre"], "AUPRC val": m["val"]["auprc"], "AUPRC prueba": t["auprc"],
                      "IC95 AUPRC": f"{m['ic_test']['auprc_ic95'][0]:.3f} a {m['ic_test']['auprc_ic95'][1]:.3f}",
                      "recall": t["recall"], "precision": t["precision"], "F1": t["f1"],
                      "VP": int(t["vp"]), "FP": int(t["fp"]), "FN": int(t["fn"]), "costo": t["costo"]})
    return pd.DataFrame(filas).round(4)

# AVANCE III · Implementación y pruebas iniciales
Se entrenan las versiones v1 de cada modelo con los valores definidos en el diseño del Avance I. La regresión logística no estaba en el diseño, se agrega como línea base para saber cuánto aportan los modelos más complejos.

## 6. Línea base: regresión logística

In [ ]:
from sklearn.linear_model import LogisticRegression
try:
    t0 = time.time()
    logreg = LogisticRegression(class_weight="balanced", max_iter=3000)
    logreg.fit(X_tr, y_tr)
    t = time.time() - t0
    registrar("logreg", "Regresión logística", logreg.predict_proba(X_va)[:, 1], logreg.predict_proba(X_te)[:, 1],
              fase="v1", tiempo_s=t, latencia_ms=latencia_ms(lambda x: logreg.predict_proba(x), X_te))
except Exception as e:
    registrar_error("logreg", e)

## 7. Rama A · XGBoost v1
Valores del diseño: 500 árboles, tasa de aprendizaje 0,05, profundidad 6, submuestreo del 80 % y `scale_pos_weight` igual a la razón de desbalance. Se agrega parada temprana sobre el AUPRC de validación.

In [ ]:
import xgboost as xgb
P_XGB_V1 = dict(n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8,
                colsample_bytree=0.8, scale_pos_weight=RAZON)

def nuevo_xgb(params, semilla=42, parada=50):
    return xgb.XGBClassifier(**params, objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
                             device=DISPOSITIVO, random_state=semilla, early_stopping_rounds=parada, n_jobs=-1)

def xgb_cpu(modelo):
    m = copy.deepcopy(modelo); m.set_params(device="cpu"); return m

try:
    t0 = time.time()
    xgb_v1 = nuevo_xgb(P_XGB_V1)
    # La ultima pareja de eval_set (validacion) es la que controla la parada temprana.
    xgb_v1.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_va, y_va)], verbose=False)
    t = time.time() - t0
    ev = xgb_v1.evals_result()
    RESULTADOS["historias"]["xgb_v1"] = {"train_aucpr": ev["validation_0"]["aucpr"], "val_aucpr": ev["validation_1"]["aucpr"]}
    cpu = xgb_cpu(xgb_v1)
    registrar("xgb_v1", "XGBoost v1", xgb_v1.predict_proba(X_va)[:, 1], xgb_v1.predict_proba(X_te)[:, 1],
              fase="v1", tiempo_s=t, mejor_iteracion=int(xgb_v1.best_iteration), params=P_XGB_V1,
              latencia_ms=latencia_ms(lambda x: cpu.predict_proba(x), X_te))
except Exception as e:
    registrar_error("xgb_v1", e)

## 8. Rama A · LightGBM v1
La alternativa de boosting propuesta en el diseño, con los mismos valores equivalentes.

In [ ]:
import lightgbm as lgb
P_LGB_V1 = dict(n_estimators=500, learning_rate=0.05, max_depth=6, num_leaves=31, subsample=0.8,
                subsample_freq=1, colsample_bytree=0.8, scale_pos_weight=RAZON)
try:
    t0 = time.time()
    lgb_v1 = lgb.LGBMClassifier(**P_LGB_V1, metric="average_precision", random_state=42, verbose=-1, n_jobs=-1)
    lgb_v1.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
               callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    t = time.time() - t0
    RESULTADOS["historias"]["lgb_v1"] = {"val_aucpr": lgb_v1.evals_result_["valid_0"]["average_precision"]}
    registrar("lgb_v1", "LightGBM v1", lgb_v1.predict_proba(X_va)[:, 1], lgb_v1.predict_proba(X_te)[:, 1],
              fase="v1", tiempo_s=t, mejor_iteracion=int(lgb_v1.best_iteration_ or P_LGB_V1["n_estimators"]),
              params=P_LGB_V1, latencia_ms=latencia_ms(lambda x: lgb_v1.predict_proba(x), X_te))
except Exception as e:
    registrar_error("lgb_v1", e)

## 9. Rama B · Definición de la red neuronal tabular
- `FocalLoss`: pérdida focal de Lin et al. (2017), le quita peso a los ejemplos fáciles para que la red se concentre en los fraudes.
- `EmbeddingPeriodico`: embeddings numéricos de Gorishniy et al. (2022). Cada variable se transforma en senos y cosenos con frecuencias aprendidas, lo que ayuda a la red a aprender relaciones con saltos bruscos. Se usa solo en el rediseño.
- `RedTabular`: bloques de capa densa, BatchNorm, ReLU y Dropout, como en la Figura 2 del Avance I.
- `entrenar_red`: entrenamiento con AdamW, parada temprana por pérdida o por AUPRC de validación, y soporte para Optuna.

In [ ]:
import torch.nn as nn

class FocalLoss(nn.Module):
    def __init__(self, alfa=0.75, gamma=2.0):
        super().__init__()
        self.alfa, self.gamma = alfa, gamma
    def forward(self, logits, y):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, y, reduction="none")
        p_t = torch.exp(-bce)                                   # probabilidad asignada a la clase correcta
        alfa_t = self.alfa * y + (1 - self.alfa) * (1 - y)
        return (alfa_t * (1 - p_t) ** self.gamma * bce).mean()

class EmbeddingPeriodico(nn.Module):
    def __init__(self, n_variables, k=8, d=16, sigma=0.5):
        super().__init__()
        self.frecuencias = nn.Parameter(torch.randn(n_variables, k) * sigma)
        limite = 1 / math.sqrt(2 * k)
        self.peso = nn.Parameter(torch.empty(n_variables, 2 * k, d).uniform_(-limite, limite))
        self.sesgo = nn.Parameter(torch.zeros(n_variables, d))
        self.n_salida = n_variables * d
    def forward(self, x):                                       # x: (lote, variables)
        v = 2 * math.pi * self.frecuencias.unsqueeze(0) * x.unsqueeze(-1)   # (lote, variables, k)
        v = torch.cat([torch.sin(v), torch.cos(v)], dim=-1)                 # (lote, variables, 2k)
        v = torch.einsum("bfi,fid->bfd", v, self.peso) + self.sesgo         # (lote, variables, d)
        return torch.relu(v).flatten(1)                                     # (lote, variables * d)

class RedTabular(nn.Module):
    def __init__(self, n_entradas, capas=(128, 64, 32), dropout=(0.3, 0.3, 0.2), embedding=None):
        super().__init__()
        self.embedding = embedding
        anterior = embedding.n_salida if embedding is not None else n_entradas
        bloques = []
        for unidades, p in zip(capas, dropout):
            bloques += [nn.Linear(anterior, unidades), nn.BatchNorm1d(unidades), nn.ReLU(), nn.Dropout(p)]
            anterior = unidades
        bloques.append(nn.Linear(anterior, 1))                 # salida: un logit, la sigmoide da la probabilidad
        self.red = nn.Sequential(*bloques)
    def forward(self, x):
        if self.embedding is not None:
            x = self.embedding(x)
        return self.red(x).squeeze(1)

Xtr_t = torch.tensor(X_tr, device=DISPOSITIVO); ytr_t = torch.tensor(y_tr, dtype=torch.float32, device=DISPOSITIVO)
Xva_t = torch.tensor(X_va, device=DISPOSITIVO); yva_t = torch.tensor(y_va, dtype=torch.float32, device=DISPOSITIVO)
Xte_t = torch.tensor(X_te, device=DISPOSITIVO)

def construir_red(cfg):
    emb = None
    if cfg.get("embedding"):
        emb = EmbeddingPeriodico(N_VARIABLES, k=cfg.get("emb_k", 8), d=cfg.get("emb_d", 16), sigma=cfg["emb_sigma"])
    return RedTabular(N_VARIABLES, cfg["capas"], cfg["dropout"], emb).to(DISPOSITIVO)

def construir_perdida(cfg):
    if cfg["perdida"] == "focal":
        return FocalLoss(cfg["alfa"], cfg["gamma"])
    return nn.BCEWithLogitsLoss(pos_weight=torch.tensor(RAZON ** cfg["exp_peso"], device=DISPOSITIVO))

def logits_por_lotes(modelo, X_t, lote=65536):
    return torch.cat([modelo(X_t[i:i + lote]) for i in range(0, len(X_t), lote)])

def predecir_red(modelo, X_t):
    modelo.eval()
    with torch.no_grad():
        return torch.sigmoid(logits_por_lotes(modelo, X_t)).float().cpu().numpy()

def entrenar_red(cfg, semilla=42, parada="val_auprc", epocas_max=None, trial=None):
    fijar_semilla(semilla)
    modelo, perdida = construir_red(cfg), construir_perdida(cfg)
    opt = torch.optim.AdamW(modelo.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    n, bs = len(Xtr_t), cfg["batch"]
    hist = {"train_loss": [], "val_loss": [], "val_auprc": [], "segundos": []}
    mejor, mejor_estado, mejor_epoca, espera = -np.inf, None, 0, 0
    for epoca in range(epocas_max or CONFIG["epocas_max"]):
        t0 = time.time()
        modelo.train()
        perm = torch.randperm(n, device=DISPOSITIVO)
        acumulado, pasos = 0.0, 0
        for i in range(0, n - bs + 1, bs):                      # se descarta el ultimo lote incompleto
            idx = perm[i:i + bs]
            opt.zero_grad(set_to_none=True)
            l = perdida(modelo(Xtr_t[idx]), ytr_t[idx])
            l.backward()
            opt.step()
            acumulado += l.item(); pasos += 1
        modelo.eval()
        with torch.no_grad():
            logits_va = logits_por_lotes(modelo, Xva_t)
            l_va = perdida(logits_va, yva_t).item()
        probs_va = torch.sigmoid(logits_va).float().cpu().numpy()
        if not np.isfinite(l_va) or not np.isfinite(probs_va).all():
            # El entrenamiento divergio (valores NaN). En Optuna se descarta el trial;
            # fuera de Optuna se conserva el mejor estado encontrado hasta ese momento.
            if trial is not None:
                raise optuna.TrialPruned()
            if mejor_estado is None:
                raise RuntimeError("La red divergio en la primera epoca")
            break
        auprc_va = float(average_precision_score(y_va, probs_va))
        hist["train_loss"].append(acumulado / pasos); hist["val_loss"].append(l_va)
        hist["val_auprc"].append(auprc_va); hist["segundos"].append(time.time() - t0)
        puntaje = auprc_va if parada == "val_auprc" else -l_va
        if puntaje > mejor:
            mejor, mejor_epoca, espera = puntaje, epoca, 0
            mejor_estado = copy.deepcopy(modelo.state_dict())
        else:
            espera += 1
        if trial is not None:
            trial.report(auprc_va, epoca)
            if trial.should_prune():
                raise optuna.TrialPruned()
        if espera >= cfg["paciencia"]:
            break
    modelo.load_state_dict(mejor_estado)
    hist["mejor_epoca"] = mejor_epoca + 1
    hist["parametros"] = int(sum(p.numel() for p in modelo.parameters()))
    return modelo, hist

def red_cpu(modelo):
    m = copy.deepcopy(modelo).cpu().eval()
    def predictor(x):
        with torch.no_grad():
            return torch.sigmoid(m(torch.from_numpy(x))).numpy()
    return predictor

## 10. Rama B · Red MLP v1
Arquitectura del diseño: capas de 128, 64 y 32 neuronas, dropout de 0,3, 0,3 y 0,2, pérdida focal con alfa 0,75 y gamma 2, AdamW con tasa de 0,001, lotes de 2.048 y parada temprana cuando la **pérdida** de validación no mejora en 8 épocas. Se registran la pérdida y el AUPRC en cada época para revisar si los dos criterios coinciden.

In [ ]:
CFG_MLP_V1 = {"capas": [128, 64, 32], "dropout": [0.3, 0.3, 0.2], "lr": 1e-3, "weight_decay": 1e-2,
              "batch": 2048, "perdida": "focal", "alfa": 0.75, "gamma": 2.0, "paciencia": 8}
try:
    t0 = time.time()
    mlp_v1, hist = entrenar_red(CFG_MLP_V1, semilla=42, parada="val_loss")
    t = time.time() - t0
    RESULTADOS["historias"]["mlp_v1"] = hist
    registrar("mlp_v1", "Red MLP v1", predecir_red(mlp_v1, Xva_t), predecir_red(mlp_v1, Xte_t),
              fase="v1", tiempo_s=t, cfg=CFG_MLP_V1, epocas=len(hist["val_loss"]), mejor_epoca=hist["mejor_epoca"],
              parametros=hist["parametros"], latencia_ms=latencia_ms(red_cpu(mlp_v1), X_te))
    print("Época elegida por pérdida:", hist["mejor_epoca"],
          "| época con mejor AUPRC:", int(np.argmax(hist["val_auprc"])) + 1)
except Exception as e:
    registrar_error("mlp_v1", e)

## 11. Rama B · TabNet v1
Segunda variante de la red, con atención secuencial sobre las variables (Arik y Pfister, 2021). Se usa muestreo balanceado de clases y parada temprana por AUPRC de validación.

In [ ]:
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    from pytorch_tabnet.metrics import Metric

    class AUPRC(Metric):
        def __init__(self):
            self._name, self._maximize = "auprc", True
        def __call__(self, y_true, y_score):
            return average_precision_score(y_true, y_score[:, 1])

    t0 = time.time()
    tabnet = TabNetClassifier(n_d=16, n_a=16, n_steps=4, gamma=1.5, lambda_sparse=1e-4, seed=42,
                              optimizer_fn=torch.optim.Adam, optimizer_params=dict(lr=2e-2),
                              scheduler_fn=torch.optim.lr_scheduler.StepLR,
                              scheduler_params=dict(step_size=10, gamma=0.9),
                              device_name=DISPOSITIVO, verbose=5)
    tabnet.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_name=["val"], eval_metric=[AUPRC],
               max_epochs=CONFIG["epocas_max"], patience=10, batch_size=4096, virtual_batch_size=512,
               weights=1, drop_last=True)
    t = time.time() - t0
    RESULTADOS["historias"]["tabnet_v1"] = {"train_loss": list(tabnet.history["loss"]),
                                            "val_auprc": list(tabnet.history["val_auprc"])}
    RESULTADOS["extra"]["importancia_tabnet"] = dict(zip(COLUMNAS, map(float, tabnet.feature_importances_)))
    registrar("tabnet_v1", "TabNet v1", tabnet.predict_proba(X_va)[:, 1], tabnet.predict_proba(X_te)[:, 1],
              fase="v1", tiempo_s=t, epocas=len(tabnet.history["loss"]))
except Exception as e:
    registrar_error("tabnet_v1", e)

## 12. Resultados de las pruebas iniciales

In [ ]:
display(tabla("v1"))

fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
paleta = {"logreg": COLORES["logreg"], "xgb_v1": COLORES["xgb"], "lgb_v1": COLORES["lgb"],
          "mlp_v1": COLORES["mlp"], "tabnet_v1": COLORES["tabnet"]}
for clave, color in paleta.items():
    if clave in PREDICCIONES:
        pr, rc, _ = precision_recall_curve(y_te, PREDICCIONES[clave]["test"])
        ejes[0].plot(rc, pr, color=color, lw=2, label=f"{NOMBRES[clave]} ({RESULTADOS['modelos'][clave]['test']['auprc']:.3f})")
ejes[0].set(xlabel="Recall", ylabel="Precisión", title="Curvas precisión-recall en prueba (v1)")
ejes[0].legend(fontsize=8); ejes[0].grid(alpha=.3)
if "mlp_v1" in RESULTADOS["historias"]:
    h = RESULTADOS["historias"]["mlp_v1"]; ep = range(1, len(h["val_loss"]) + 1)
    ejes[1].plot(ep, h["val_auprc"], color=COLORES["mlp"], lw=2, label="AUPRC validación")
    ejes[1].axvline(h["mejor_epoca"], color=COLORES["logreg"], ls="--", label="época elegida por pérdida")
    ejes[1].set(xlabel="Época", ylabel="AUPRC", title="Red MLP v1: AUPRC de validación por época")
    ejes[1].legend(fontsize=8); ejes[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{SALIDA}/v1_resumen.png", dpi=150); plt.show()

## 13. Análisis de errores
Importancia de variables con SHAP nativo de XGBoost, y revisión de los fraudes que el mejor modelo v1 deja pasar.

In [ ]:
try:
    booster = xgb_v1.get_booster()
    gan = booster.get_score(importance_type="gain")
    RESULTADOS["extra"]["importancia_xgb_gain"] = {COLUMNAS[int(k[1:])]: float(v) for k, v in gan.items()}
    rng = np.random.default_rng(42)
    idx = np.unique(np.concatenate([rng.choice(len(X_te), 3000, replace=False), np.where(y_te == 1)[0]]))
    contrib = booster.predict(xgb.DMatrix(X_te[idx]), pred_contribs=True,
                              iteration_range=(0, int(xgb_v1.best_iteration) + 1))
    shap_v = contrib[:, :-1]
    RESULTADOS["extra"]["shap_media_abs"] = dict(zip(COLUMNAS, map(float, np.abs(shap_v).mean(0))))
    np.savez_compressed(f"{SALIDA}/shap_muestra.npz", shap=shap_v.astype(np.float32),
                        X=X_te[idx], y=y_te[idx], columnas=np.array(COLUMNAS))
    print(pd.Series(RESULTADOS["extra"]["shap_media_abs"]).sort_values(ascending=False).head(10).round(3))
except Exception as e:
    registrar_error("shap", e)

try:
    v1 = {k: m for k, m in RESULTADOS["modelos"].items() if m.get("fase") == "v1"}
    mejor = max(v1, key=lambda k: v1[k]["val"]["auprc"])
    u = v1[mejor]["test"]["umbral"]
    p = PREDICCIONES[mejor]["test"]
    fraudes = pd.DataFrame({"monto": montos_te, "hora": hora_te, "prob": p, "y": y_te})[y_te == 1]
    fraudes["detectado"] = fraudes["prob"] >= u
    resumen = fraudes.groupby("detectado").agg(casos=("monto", "size"), monto_mediana=("monto", "median"),
                                                monto_total=("monto", "sum"), prob_mediana=("prob", "median"))
    RESULTADOS["extra"]["analisis_fn"] = {"modelo": mejor, "umbral": u,
                                          "resumen": json.loads(resumen.reset_index().to_json(orient="records")),
                                          "fraudes": json.loads(fraudes.round(5).to_json(orient="records"))}
    guardar()
    print("Mejor modelo v1 en validación:", NOMBRES[mejor]); display(resumen)
except Exception as e:
    registrar_error("analisis_fn", e)

# AVANCE IV · Rediseño y resultados
A partir de lo observado en las pruebas iniciales se aplican cuatro cambios:

1. **Ajuste de hiperparámetros con Optuna**, con un presupuesto comparable para XGBoost y para la red.
2. **Parada temprana por AUPRC** en lugar de pérdida, y búsqueda del tipo de pérdida (focal o BCE ponderada).
3. **Embeddings numéricos periódicos** para la red, que atacan la debilidad de las redes con funciones irregulares descrita por Grinsztajn et al. (2022).
4. **Varias semillas y ensamble** entre la mejor red y XGBoost, para medir la variabilidad y aprovechar que los dos modelos se equivocan en casos distintos.

Todas las decisiones se toman con validación, el conjunto de prueba solo se usa para medir.

## 14. Ajuste de XGBoost con Optuna

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objetivo_xgb(trial):
    params = dict(
        n_estimators=2000,
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        min_child_weight=trial.suggest_float("min_child_weight", 1, 20, log=True),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        gamma=trial.suggest_float("gamma", 0.0, 5.0),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        scale_pos_weight=RAZON ** trial.suggest_float("exp_peso", 0.0, 1.0))
    m = nuevo_xgb(params, parada=100)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    trial.set_user_attr("mejor_iteracion", int(m.best_iteration))
    return float(average_precision_score(y_va, m.predict_proba(X_va)[:, 1]))

try:
    t0 = time.time()
    estudio_xgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    estudio_xgb.optimize(objetivo_xgb, n_trials=CONFIG["trials_xgb"], show_progress_bar=True)
    estudio_xgb.trials_dataframe().to_csv(f"{SALIDA}/optuna_xgb.csv", index=False)
    RESULTADOS["extra"]["optuna_xgb"] = {"mejor_valor": estudio_xgb.best_value, "mejores_params": estudio_xgb.best_params,
                                         "trials": len(estudio_xgb.trials), "minutos": (time.time() - t0) / 60}
    guardar()
    print("Mejor AUPRC de validación:", round(estudio_xgb.best_value, 4)); print(estudio_xgb.best_params)
except Exception as e:
    registrar_error("optuna_xgb", e)

## 15. Ajuste de la red con Optuna
Se buscan arquitectura, regularización, tamaño de lote y tipo de pérdida. Los trials que van por debajo de la mediana se cortan antes de terminar. Si un trial diverge y produce valores NaN, se descarta sin detener el estudio (corrección hecha después de la primera ejecución, en la que un trial divergente detuvo la búsqueda).

In [ ]:
def objetivo_red(trial):
    ancho = trial.suggest_categorical("ancho", [64, 128, 256])
    n_capas = trial.suggest_int("n_capas", 2, 4)
    drop = trial.suggest_float("dropout", 0.0, 0.5)
    cfg = {"capas": [max(ancho // (2 ** i), 16) for i in range(n_capas)], "dropout": [drop] * n_capas,
           "lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True),
           "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-1, log=True),
           "batch": trial.suggest_categorical("batch", [1024, 2048, 4096]),
           "perdida": trial.suggest_categorical("perdida", ["focal", "bce_ponderada"]), "paciencia": 6}
    if cfg["perdida"] == "focal":
        cfg["alfa"] = trial.suggest_float("alfa", 0.25, 0.95)
        cfg["gamma"] = trial.suggest_float("gamma", 0.5, 3.0)
    else:
        cfg["exp_peso"] = trial.suggest_float("exp_peso", 0.0, 1.0)
    trial.set_user_attr("cfg", cfg)
    _, hist = entrenar_red(cfg, semilla=42, parada="val_auprc", epocas_max=CONFIG["epocas_max_optuna"], trial=trial)
    trial.set_user_attr("epocas", len(hist["val_auprc"]))
    return max(hist["val_auprc"])

try:
    t0 = time.time()
    estudio_red = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42),
                                      pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=6))
    estudio_red.optimize(objetivo_red, n_trials=CONFIG["trials_red"],
                         timeout=CONFIG["minutos_max_optuna_red"] * 60, show_progress_bar=True,
                         catch=(RuntimeError, ValueError))
    estudio_red.trials_dataframe().to_csv(f"{SALIDA}/optuna_red.csv", index=False)
    CFG_MLP_V2 = dict(estudio_red.best_trial.user_attrs["cfg"], paciencia=8)
    RESULTADOS["extra"]["optuna_red"] = {"mejor_valor": estudio_red.best_value, "mejores_params": estudio_red.best_params,
                                         "cfg": CFG_MLP_V2, "trials": len(estudio_red.trials),
                                         "podados": sum(t.state == optuna.trial.TrialState.PRUNED for t in estudio_red.trials),
                                         "minutos": (time.time() - t0) / 60}
    guardar()
    print("Mejor AUPRC de validación:", round(estudio_red.best_value, 4)); print(CFG_MLP_V2)
except Exception as e:
    registrar_error("optuna_red", e)
    CFG_MLP_V2 = dict(CFG_MLP_V1)

## 16. Red MLP v2 con varias semillas
La configuración ganadora se entrena con tres semillas. La predicción final es el promedio de las tres, y la dispersión entre semillas muestra qué tan estable es la red.

In [ ]:
def red_varias_semillas(cfg, clave, nombre):
    p_va, p_te, por_semilla, modelos = [], [], [], []
    t0 = time.time()
    for s in CONFIG["semillas_finales"]:
        modelo, hist = entrenar_red(cfg, semilla=s, parada="val_auprc")
        pv, pt = predecir_red(modelo, Xva_t), predecir_red(modelo, Xte_t)
        p_va.append(pv); p_te.append(pt); modelos.append(modelo)
        por_semilla.append({"semilla": s, "val_auprc": float(average_precision_score(y_va, pv)),
                            "test_auprc": float(average_precision_score(y_te, pt)), "epocas": len(hist["val_auprc"]),
                            "mejor_epoca": hist["mejor_epoca"]})
        if s == CONFIG["semillas_finales"][0]:
            RESULTADOS["historias"][clave] = hist
    t = time.time() - t0
    torch.save(modelos[0].state_dict(), f"{SALIDA}/{clave}_semilla{CONFIG['semillas_finales'][0]}.pt")
    registrar(clave, nombre, np.mean(p_va, 0), np.mean(p_te, 0), fase="v2", tiempo_s=t, cfg=cfg,
              semillas=por_semilla, parametros=hist["parametros"],
              latencia_ms=latencia_ms(red_cpu(modelos[0]), X_te) * len(modelos))
    print(pd.DataFrame(por_semilla))
    return modelos

try:
    mlp_v2 = red_varias_semillas(CFG_MLP_V2, "mlp_v2", "Red MLP v2")
except Exception as e:
    registrar_error("mlp_v2", e)

## 17. Red MLP v2 con embeddings periódicos
Se prueba la escala inicial de las frecuencias (sigma) con una semilla, se escoge la mejor en validación y se entrena con las tres semillas.

In [ ]:
try:
    busqueda = []
    for sigma in CONFIG["sigmas_embedding"]:
        cfg = dict(CFG_MLP_V2, embedding=True, emb_k=8, emb_d=16, emb_sigma=sigma)
        _, hist = entrenar_red(cfg, semilla=42, parada="val_auprc")
        busqueda.append({"sigma": sigma, "val_auprc": max(hist["val_auprc"]), "epocas": len(hist["val_auprc"])})
        print(busqueda[-1])
    RESULTADOS["extra"]["busqueda_sigma"] = busqueda
    sigma = max(busqueda, key=lambda r: r["val_auprc"])["sigma"]
    CFG_EMB = dict(CFG_MLP_V2, embedding=True, emb_k=8, emb_d=16, emb_sigma=sigma)
    mlp_emb = red_varias_semillas(CFG_EMB, "mlp_emb", "Red MLP v2 + embeddings")
except Exception as e:
    registrar_error("mlp_emb", e)

## 18. XGBoost v2 con varias semillas
Mismo tratamiento que la red, para que la comparación sea justa.

In [ ]:
try:
    mp = dict(estudio_xgb.best_params)
    exp_peso = mp.pop("exp_peso")
    P_XGB_V2 = dict(mp, n_estimators=2000, scale_pos_weight=RAZON ** exp_peso)
    p_va, p_te, por_semilla, modelos = [], [], [], []
    t0 = time.time()
    for s in CONFIG["semillas_finales"]:
        m = nuevo_xgb(P_XGB_V2, semilla=s, parada=100)
        m.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_va, y_va)], verbose=False)
        pv, pt = m.predict_proba(X_va)[:, 1], m.predict_proba(X_te)[:, 1]
        p_va.append(pv); p_te.append(pt); modelos.append(m)
        por_semilla.append({"semilla": s, "val_auprc": float(average_precision_score(y_va, pv)),
                            "test_auprc": float(average_precision_score(y_te, pt)), "mejor_iteracion": int(m.best_iteration)})
        if s == CONFIG["semillas_finales"][0]:
            ev = m.evals_result()
            RESULTADOS["historias"]["xgb_v2"] = {"train_aucpr": ev["validation_0"]["aucpr"], "val_aucpr": ev["validation_1"]["aucpr"]}
    t = time.time() - t0
    modelos[0].save_model(f"{SALIDA}/xgb_v2_semilla{CONFIG['semillas_finales'][0]}.json")
    cpu = xgb_cpu(modelos[0])
    registrar("xgb_v2", "XGBoost v2", np.mean(p_va, 0), np.mean(p_te, 0), fase="v2", tiempo_s=t,
              params={k: float(v) for k, v in P_XGB_V2.items()}, semillas=por_semilla,
              latencia_ms=latencia_ms(lambda x: cpu.predict_proba(x), X_te) * len(modelos))
    print(pd.DataFrame(por_semilla))
except Exception as e:
    registrar_error("xgb_v2", e)

## 19. Ensamble XGBoost + red
Se promedian los percentiles de cada modelo en lugar de las probabilidades, porque los dos modelos producen probabilidades en escalas distintas.

In [ ]:
def percentil(p):
    return pd.Series(p).rank(pct=True).to_numpy()

try:
    redes = [k for k in ("mlp_v2", "mlp_emb") if k in RESULTADOS["modelos"]]
    mejor_red = max(redes, key=lambda k: RESULTADOS["modelos"][k]["val"]["auprc"])
    pv = (percentil(PREDICCIONES["xgb_v2"]["val"]) + percentil(PREDICCIONES[mejor_red]["val"])) / 2
    pt = (percentil(PREDICCIONES["xgb_v2"]["test"]) + percentil(PREDICCIONES[mejor_red]["test"])) / 2
    registrar("ensamble", f"Ensamble XGBoost + {NOMBRES[mejor_red].replace('Red ', '')}", pv, pt,
              fase="v2", componentes=["xgb_v2", mejor_red])
    # Cuantos fraudes detecta cada modelo que el otro no ve
    ux, ur = RESULTADOS["modelos"]["xgb_v2"]["test"]["umbral"], RESULTADOS["modelos"][mejor_red]["test"]["umbral"]
    dx = PREDICCIONES["xgb_v2"]["test"][y_te == 1] >= ux
    dr = PREDICCIONES[mejor_red]["test"][y_te == 1] >= ur
    RESULTADOS["extra"]["solapamiento"] = {"red": mejor_red, "ambos": int((dx & dr).sum()), "solo_xgb": int((dx & ~dr).sum()),
                                           "solo_red": int((~dx & dr).sum()), "ninguno": int((~dx & ~dr).sum())}
    guardar(); print(RESULTADOS["extra"]["solapamiento"])
except Exception as e:
    registrar_error("ensamble", e)

## 20. Comparación final y prueba estadística
El bootstrap pareado responde la pregunta del proyecto: si el intervalo de la diferencia de AUPRC incluye el cero, no hay evidencia de que un modelo sea mejor que el otro.

In [ ]:
try:
    comparaciones = {}
    pares = [("xgb_v1", "mlp_v1"), ("xgb_v2", "mlp_v2"), ("xgb_v2", "mlp_emb"), ("ensamble", "xgb_v2"),
             ("xgb_v2", "xgb_v1"), ("mlp_v2", "mlp_v1"), ("mlp_emb", "mlp_v2"), ("xgb_v1", "logreg")]
    for a, b in pares:
        if a in PREDICCIONES and b in PREDICCIONES:
            comparaciones[f"{a} vs {b}"] = bootstrap_pareado(y_te, PREDICCIONES[a]["test"], PREDICCIONES[b]["test"])
    RESULTADOS["extra"]["comparaciones"] = comparaciones
    guardar()
    display(tabla())
    display(pd.DataFrame(comparaciones).T.round(4))
except Exception as e:
    registrar_error("comparacion", e)

try:
    fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
    paleta = {"xgb_v2": COLORES["xgb"], "mlp_v2": COLORES["mlp"], "mlp_emb": COLORES["emb"], "ensamble": COLORES["ens"]}
    for clave, color in paleta.items():
        if clave in PREDICCIONES:
            pr, rc, _ = precision_recall_curve(y_te, PREDICCIONES[clave]["test"])
            ejes[0].plot(rc, pr, color=color, lw=2, label=f"{NOMBRES[clave]} ({RESULTADOS['modelos'][clave]['test']['auprc']:.3f})")
    ejes[0].set(xlabel="Recall", ylabel="Precisión", title="Curvas precisión-recall en prueba (v2)")
    ejes[0].legend(fontsize=8); ejes[0].grid(alpha=.3)
    claves = [k for k in RESULTADOS["modelos"]]
    medias = [RESULTADOS["modelos"][k]["test"]["auprc"] for k in claves]
    ic = np.array([RESULTADOS["modelos"][k]["ic_test"]["auprc_ic95"] for k in claves])
    medias = np.array(medias)
    ejes[1].errorbar(medias, range(len(claves)), xerr=[np.clip(medias - ic[:, 0], 0, None), np.clip(ic[:, 1] - medias, 0, None)],
                     fmt="o", color=COLORES["xgb"], ecolor=COLORES["logreg"], capsize=3)
    ejes[1].set_yticks(range(len(claves))); ejes[1].set_yticklabels([NOMBRES[k] for k in claves], fontsize=8)
    ejes[1].set(xlabel="AUPRC en prueba con IC 95 %", title="Todos los modelos"); ejes[1].grid(alpha=.3)
    plt.tight_layout(); plt.savefig(f"{SALIDA}/v2_resumen.png", dpi=150); plt.show()
except Exception as e:
    registrar_error("grafica_final", e)

## 21. Exportar resultados
Se empaqueta todo en `resultados_avances_3_4.zip` y se descarga. Ese archivo es el que se usa para el informe. Después de esto conviene descargar también el notebook ya ejecutado (*Archivo → Descargar → .ipynb*) para subirlo al repositorio con sus salidas.

In [ ]:
import sklearn
versiones = {"python": platform.python_version(), "sklearn": sklearn.__version__, "numpy": np.__version__}
for lib in ["torch", "xgboost", "lightgbm", "optuna", "pytorch_tabnet"]:
    try:
        versiones[lib] = __import__(lib).__version__
    except Exception:
        versiones[lib] = "no disponible"
RESULTADOS["extra"]["entorno"] = {"dispositivo": DISPOSITIVO, "gpu": GPU, "versiones": versiones,
                                  "minutos_totales": (time.time() - INICIO) / 60, "modo_rapido": MODO_RAPIDO}
np.savez_compressed(f"{SALIDA}/etiquetas.npz", y_val=y_va, y_test=y_te, montos_test=montos_te, hora_test=hora_te)
guardar()
archivo = shutil.make_archive("/content/resultados_avances_3_4" if os.path.isdir("/content") else "resultados_avances_3_4",
                              "zip", SALIDA)
print("Listo en", round(RESULTADOS["extra"]["entorno"]["minutos_totales"], 1), "minutos:", archivo)
if RESULTADOS["errores"]:
    print("Hubo errores en:", list(RESULTADOS["errores"]))
try:
    from google.colab import files
    files.download(archivo)
except Exception:
    print("Descargue el archivo manualmente desde el panel de archivos")